In [6]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import math as m
import pyspark.sql.functions as F

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

In [14]:
temp = spark.read.parquet("../taxi_data/irregular")

temp = temp.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")\
    .drop("store_and_fwd_flag", "RateCodeID", "passenger_count", "payment_type", "mta_tax", "tolls_amount", "improvement_surcharge", "extra", "trip_type", "ehail_fee")

temp_work = temp.filter(F.col("pickup_datetime").isNotNull() & F.col("dropoff_datetime").isNotNull()) \
    .filter(F.col("pickup_datetime") < F.col("dropoff_datetime"))

In [17]:
temp_work.count()

66088

In [24]:
temp2 = spark.read.parquet("../taxi_data/yellow/2025")

temp2.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-02-01 00:12:18|  2025-02-01 00:32:33|              3|         3.12|         1|                 N|         246|    

In [ ]:
download all the data from online
put all library in requirements.txt
xinyul27@unimelb.edu.au


# predict time series for 2024 2 months and then predict next month
# do this recursively and update hyperparameters of model until the model is good enough
# to predict 2025 march given 2025 january and february data

# also aggregate the data to mean congestion time per day(based on pickup_datetime) and sort in order to plot the time series



In [1]:
import lightgbm as lgb
base_model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05)
model = MultiOutputRegressor(base_model)

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:
# Convert predictions
preds_df = pd.DataFrame(validation_preds)

# Align with y_val
y_val_aligned = X_val[["DOLocationID", "date"]].merge(
    y_val.reset_index(drop=True),
    left_index=True,
    right_index=True
)

# Merge on ID + date
eval_df = preds_df.merge(y_val_aligned, on=["DOLocationID", "date"], suffixes=("_pred", "_true"))


print("R^2:", r2_score(eval_df["median_congestion_time_seconds_true"], eval_df["median_congestion_time_seconds_pred"]))
print("RMSE:", root_mean_squared_error(eval_df["median_congestion_time_seconds_true"], eval_df["median_congestion_time_seconds_pred"]))